In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =================================================
# USER SETTINGS
# =================================================
BASE_DIR = os.getcwd()   # Run script from the Trieste directory
OUTPUT_EXCEL = "Trieste_Error_Metrics_All_Emulators.xlsx"
TEST_YEARS = [1995, 1996, 2020]
PERC = np.arange(0, 101, 1)

# =================================================
# HELPER FUNCTIONS
# =================================================
def extract_run_number(filename):
    """
    Extract run number from filenames like:
    lstmHybridModel_Run12_testresults.txt
    """
    return int(filename.split("Run")[1].split("_")[0])


def compute_metrics(file_path):
    """
    Compute performance metrics for one run
    """
    dyn = np.loadtxt(file_path)

    # Filter test years
    idx_test = np.isin(dyn[:, 0], TEST_YEARS)
    dynTest = dyn[idx_test]

    # Remove mean
    x = dynTest[:, 8] - np.nanmean(dynTest[:, 8])
    y = dynTest[:, 7] - np.nanmean(dynTest[:, 7])

    # Remove NaNs
    valid = ~np.isnan(x)
    xr = x[valid]
    yr = y[valid]

    # Linear regression (OLS)
    X = sm.add_constant(xr)
    model = sm.OLS(yr, X).fit()
    slope = model.params[1]

    # Metrics
    slf = slope
    corr = np.corrcoef(xr, yr)[0, 1]
    rmse = np.sqrt(np.mean((xr - yr) ** 2))
    mad = np.mean(np.abs(xr - yr))

    # MADp and MADc
    prc_x = np.percentile(xr, PERC)
    prc_y = np.percentile(yr, PERC)
    madp = np.mean(np.abs(prc_y - prc_x))
    madc = mad + madp

    return slf, corr, rmse, mad, madp, madc


# =================================================
# MAIN PROCESSING LOOP
# =================================================
writer = pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl")

for emulator_folder in sorted(os.listdir(BASE_DIR)):

    folder_path = os.path.join(BASE_DIR, emulator_folder)

    # Skip non-directories
    if not os.path.isdir(folder_path):
        continue

    # Collect test result files
    run_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".txt") and "testresults" in f
    ]

    if len(run_files) == 0:
        continue

    # Sort runs numerically
    run_files = sorted(run_files, key=extract_run_number)

    print(f"Processing {emulator_folder} ({len(run_files)} runs)")

    results = []
    run_ids = []

    for run_file in run_files:
        file_path = os.path.join(folder_path, run_file)
        metrics = compute_metrics(file_path)
        results.append(metrics)
        run_ids.append(extract_run_number(run_file))

    # Create DataFrame
    df = pd.DataFrame(
        results,
        columns=["SLF", "Corr", "RMSE", "MAD", "MADp", "MADc"],
        index=run_ids
    )
    df.index.name = "Run"

    # Excel sheet name (max 31 chars)
    sheet_name = emulator_folder[:31]
    df.to_excel(writer, sheet_name=sheet_name, float_format="%.4f")

# Save Excel file
writer.close()

print(f"\nAll results saved to: {OUTPUT_EXCEL}")


Processing LSTM-MADc (40 runs)
Processing LSTM-MSE (40 runs)
Processing LSTMh-MADc (40 runs)
Processing LSTMh-MSE (40 runs)
Processing MLP-MADc (40 runs)
Processing MLP-MSE (40 runs)
Processing MLR-MADc (40 runs)
Processing MLR-MSE (40 runs)
Processing RNN-MADc (40 runs)
Processing RNN-MSE (40 runs)
Processing RNNh-MADc (40 runs)
Processing RNNh-MSE (40 runs)

All results saved to: Trieste_Error_Metrics_All_Emulators.xlsx
